# Knopp Drive tutorial

End-to-end walkthrough of the Knopp Drive composite warp engineering bound, the headline IP of the Systrophē framework. We construct a drive, inspect its instantaneous budget, run an Earth-Mars equivalent journey, and verify hardware-confirmed band-gating extinction.

**Prerequisites**
```
pip install systrophe matplotlib numpy
```

## 1. Construct a Knopp Drive

`KnoppDrive` is the high-level interface. Its four parameters map one-to-one onto the four composing mechanisms:

| Parameter | Mechanism |
|---|---|
| `omega`, `R_cylinder`, `alpha_wall` | Tipler seed + Krasnikov tube |
| `Q` | Cavity feedback (sustained drive power $\sim 1/Q^2$) |
| `epsilon_horn`, `theta_0_horn` | Horn-toroidal steering |


In [ ]:
from systrophe.knopp_drive import KnoppDrive

drive = KnoppDrive(
    Q=100.0,
    epsilon_horn=0.2,
    theta_0_horn=0.0,
    omega=1.0,
    R_cylinder=1.0,
)
print(drive)

## 2. Inspect the instantaneous budget

`drive.budget(r_orbit)` returns a `KnoppDriveBudget` with every engineering number. Inside the Tipler CTC band the composite exotic-matter requirement is exactly zero.

In [ ]:
for r in (1.5, 3.0, 6.0):
    b = drive.budget(r_orbit=r)
    print(f"r={r}:")
    print(f"  inside_band = {drive.is_inside_band(r_orbit=r)}")
    print(f"  tipler_gate_factor = {b.tipler_gate_factor:.4f}")
    print(f"  composite |E_neg|  = {abs(b.composite_E_neg):.4e}")
    print(f"  P_drive           = {b.sustained_drive_power:.4e}")
    print()

## 3. Run an Earth-Mars equivalent journey

The Earth-Mars distance at closest approach is ~0.52 AU; in our geometric units we take 1 unit = 1 AU. The journey lies *entirely inside the first Tipler CTC band* of a unit supercritical cylinder, so the composite exotic-matter requirement is exactly zero.

In [ ]:
report = drive.journey(distance=0.52)
print(f"distance              = {report.distance}")
print(f"inside_band_fraction = {report.inside_band_fraction:.2f}")
print(f"|E_neg| total        = {report.exotic_matter_total}")
print(f"coord time           = {report.coord_time_total:.4f}")
print(f"sustained drive P    = {report.sustained_drive_power:.4e}")
print(f"total energy budget  = {report.total_energy_budget:.4e}")
print(f"Pfenning-Ford OK?    = {report.pfenning_ford_compatible}")

## 4. Steering vector

The horn-twist axis `theta_0` sets the steering direction; `epsilon_horn` sets the magnitude.

In [ ]:
import math

for theta_0 in (0.0, math.pi/2, math.pi):
    drive2 = KnoppDrive(Q=100.0, epsilon_horn=0.3, theta_0_horn=theta_0)
    p_x, p_y = drive2.steering_vector()
    mag = math.hypot(p_x, p_y)
    angle = math.atan2(p_y, p_x)
    print(f"theta_0 = {theta_0:.4f} rad: |p| = {mag:.4f}, arg(p) = {angle:.4f} rad")

## 5. Pfenning-Ford compatibility

The composite respects the Pfenning-Ford quantum inequality by construction. At very high Q over long journeys, the cavity time exceeds the bound and P-F can fail.

In [ ]:
for Q in (10.0, 100.0, 1000.0):
    d = KnoppDrive(Q=Q)
    for L in (5.0, 15.0):
        ok = d.is_pfenning_ford_compatible(distance=L)
        print(f"Q={Q:6.0f}  L={L:5.1f}  P-F ok? {ok}")

## 6. Visualise the Tipler gate factor

Plot the gate factor across the supercritical LP exterior to see the CTC bands.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from systrophe.tipler_krasnikov_hybrid import tipler_tilt_at
from systrophe.vanstockum import VanStockumInterior

vs = VanStockumInterior(omega=1.0, R=1.0)
rs = np.linspace(1.05, 12.0, 400)
gate = np.clip(1.0 - np.array([tipler_tilt_at(vs, float(r)) for r in rs]), 0, 1)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(rs, gate, color='C0')
ax.fill_between(rs, 0, 1, where=(gate == 0), color='C0', alpha=0.15,
                  label='inside CTC band')
ax.set_xlabel('orbit radius r')
ax.set_ylabel('Tipler gate factor')
ax.set_title('Tipler CTC-band gating ($\\omega = 1, R = 1$)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 7. Hardware confirmation on `ibm_marrakesh`

The Knopp Drive's headline shortcut was confirmed on IBM Quantum's 156-qubit Heron-r2 processor (Marrakesh batch 6, job `d8183b7oha1c73bk1n60`, 2026-05-11). See `experiments/marrakesh_batch_6_knopp_drive.py` for the full circuit and `experiments/results/marrakesh_batch6_hw_analysis.json` for the published results.

Hardware reproduces the simulator prediction to TV distance ≤ 0.05 at every $r$, and the catcher flags the band exit (`r3 -> r4`) as a sharp Hamming transition with step=12.

In [ ]:
import json
from pathlib import Path

hw_path = Path('../experiments/results/marrakesh_batch6_hw_analysis.json')
if hw_path.exists():
    hw = json.loads(hw_path.read_text())
    print('HW catcher verdict:', hw['novelty_catcher']['verdict'])
    print()
    print(f"{'r':6s} {'gate':6s} {'sim_P':8s} {'hw_P':8s} {'TV':6s}")
    for c in hw['per_circuit']:
        print(f"{c['r']:6.3f} {c['tipler_gate_factor']:6.3f} "
              f"{c['P_data1_predicted']:8.4f} {c['P_data1_observed']:8.4f} "
              f"{c['tv_obs_vs_pred']:6.4f}")
else:
    print('Run from inside the systrophe repo to see the HW table.')

## Further reading

- `paper/knopp_drive.pdf` — 11-page whitepaper with full GR derivations, figures, comparison table, and the HW section
- `docs/knopp_drive_api.md` — API reference
- `examples/knopp_drive_walkthrough.py` — six-configuration demo
- `examples/warp_drive_comparison.py` — Knopp Drive vs Alcubierre / Krasnikov / Lentz / Bobrick-Martire
- `experiments/marrakesh_batch_6_knopp_drive.py` — hardware experiment source